In [3]:
import pandas as pd
import numpy as np
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)
print("Shape:", df.shape)
print(df.head())

Shape: (891, 12)
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   

In [5]:
# ---------- STEP 2: MISSING DATA (5/20% RULE) ----------

missing_pct = df.isnull().mean() * 100
print("Missing % per column:\n", missing_pct)

for col in df.columns:
    pct = missing_pct[col]
    if pct == 0:
        continue
    elif pct < 5:
        df = df.dropna(subset=[col])
        print(f"{col}: <5% missing -> dropped rows")
    elif pct <= 20:
        if df[col].dtype in ['float64', 'int64']:
            df[col] = df[col].fillna(df[col].median())
            print(f"{col}: 5-20% missing -> filled with median")
        else:
            df[col] = df[col].fillna(df[col].mode()[0])
            print(f"{col}: 5-20% missing -> filled with mode")
    else:
        print(f"{col}: >20% missing -> will drop this column entirely")
        df = df.drop(columns=[col])

print("\nMissing values after cleaning:\n", df.isnull().sum())
print("\nNew shape:", df.shape)

Missing % per column:
 PassengerId    0.0
Survived       0.0
Pclass         0.0
Name           0.0
Sex            0.0
Age            0.0
SibSp          0.0
Parch          0.0
Ticket         0.0
Fare           0.0
Embarked       0.0
dtype: float64

Missing values after cleaning:
 PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64

New shape: (889, 11)


In [6]:
# ---------- STEP 3: OUTLIER HANDLING (IQR CAPPING) ----------

def cap_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[column] = df[column].clip(lower, upper)
    print(f"{column}: capped between {lower:.2f} and {upper:.2f}")
    return df

df = cap_outliers_iqr(df, 'Fare')
df = cap_outliers_iqr(df, 'Age')

print("\nSummary stats after capping:\n", df[['Fare', 'Age']].describe())

Fare: capped between -26.76 and 65.66
Age: capped between 2.50 and 54.50

Summary stats after capping:
              Fare         Age
count  889.000000  889.000000
mean    23.956061   29.000562
std     20.414997   12.051609
min      0.000000    2.500000
25%      7.895800   22.000000
50%     14.454200   28.000000
75%     31.000000   35.000000
max     65.656300   54.500000


In [7]:
# ---------- STEP 4: ONE-HOT ENCODING ----------

print("Columns before encoding:", df.columns.tolist())

df_encoded = pd.get_dummies(df, columns=['Sex', 'Embarked'], drop_first=True)

print("\nColumns after encoding:", df_encoded.columns.tolist())
print("\n", df_encoded.head())

Columns before encoding: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Embarked']

Columns after encoding: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Sex_male', 'Embarked_Q', 'Embarked_S']

    PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name   Age  SibSp  Parch  \
0                            Braund, Mr. Owen Harris  22.0      1      0   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  38.0      1      0   
2                             Heikkinen, Miss. Laina  26.0      0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  35.0      1      0   
4                           Allen, Mr. William Henry  35.0      0      0   

             Ticket     Fare  Sex_mal

In [8]:
# ---------- STEP 5: REMOVE MULTICOLLINEARITY ----------

import numpy as np

# Drop non-numeric/ID columns that don't help correlation analysis
df_model = df_encoded.drop(columns=['Name', 'Ticket', 'PassengerId'])

corr_matrix = df_model.select_dtypes(include=[np.number]).corr().abs()
upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

to_drop = [col for col in upper_triangle.columns if any(upper_triangle[col] > 0.80)]
print("Columns dropped due to high correlation (>0.80):", to_drop)

df_model = df_model.drop(columns=to_drop)

# ---------- STEP 6: FEATURE ENGINEERING (3 NEW FEATURES) ----------

df_model['FamilySize'] = df_model['SibSp'] + df_model['Parch'] + 1
df_model['IsAlone'] = (df_model['FamilySize'] == 1).astype(int)
df_model['FarePerPerson'] = df_model['Fare'] / df_model['FamilySize']

print("\nFinal columns:", df_model.columns.tolist())
print("\nFinal shape:", df_model.shape)
print(df_model.head())

Columns dropped due to high correlation (>0.80): []

Final columns: ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Sex_male', 'Embarked_Q', 'Embarked_S', 'FamilySize', 'IsAlone', 'FarePerPerson']

Final shape: (889, 12)
   Survived  Pclass   Age  SibSp  Parch     Fare  Sex_male  Embarked_Q  \
0         0       3  22.0      1      0   7.2500      True       False   
1         1       1  38.0      1      0  65.6563     False       False   
2         1       3  26.0      0      0   7.9250     False       False   
3         1       1  35.0      1      0  53.1000     False       False   
4         0       3  35.0      0      0   8.0500      True       False   

   Embarked_S  FamilySize  IsAlone  FarePerPerson  
0        True           2        0        3.62500  
1       False           2        0       32.82815  
2        True           1        1        7.92500  
3        True           2        0       26.55000  
4        True           1        1        8.05000  


In [9]:
# ---------- STEP 7: DATA VALIDATION CHECKS ----------

assert df_model['Age'].min() >= 0, "Age cannot be negative"
assert df_model.isnull().sum().sum() == 0, "There should be no missing values left"
assert df_model['Fare'].min() >= 0, "Fare cannot be negative"

print("✅ All data validation checks passed!")
print(f"\nFinal dataset ready for modeling: {df_model.shape[0]} rows, {df_model.shape[1]} columns")

✅ All data validation checks passed!

Final dataset ready for modeling: 889 rows, 12 columns


## Project Summary — Advanced EDA & Feature Engineering

**Dataset:** Titanic passenger dataset (891 rows, 12 original columns)

**Missing Data Handling:**
- Applied a threshold-based rule: <5% missing → dropped rows, 5-20% missing → statistical imputation (median for numeric, mode for categorical), >20% missing → dropped column entirely (Cabin was dropped, ~77% missing)

**Outlier Treatment:**
- Used IQR-based capping (not deletion) on Fare and Age to preserve data volume while neutralizing extreme values

**Categorical Encoding:**
- Applied One-Hot Encoding (not Label Encoding) on Sex and Embarked to avoid introducing false numeric relationships between categories

**Multicollinearity Check:**
- Built a correlation matrix and checked for feature pairs above 0.80 correlation threshold — none found, indicating the retained features are largely independent

**Feature Engineering (3 new features):**
- FamilySize: total family members aboard (SibSp + Parch + 1)
- IsAlone: binary flag for solo travelers
- FarePerPerson: fare divided by family size, revealing per-person spending patterns

**Validation:**
- Confirmed no negative ages or fares, and zero remaining missing values before finalizing the dataset

**Final Output:** A clean, validated dataset (889 rows × 12 columns) ready for downstream machine learning modeling.